# Deep Clustering (SimCLR / BYOL) для СЭМ-изображений

### Шаг 1: Подключите GPU
`Среда выполнения` -> `Сменить среду выполнения` -> T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === Код из GitHub + данные из Drive ===
import os

# 1. Клонируем / обновляем код
if os.path.exists('/content/diploma_project/.git'):
    !cd /content/diploma_project && git pull
else:
    !git clone https://github.com/daniilctrl/sem-image-analysis.git /content/diploma_project

# 2. Распаковываем данные из zip (только если ещё не распакованы)
if not os.path.exists('/content/diploma_project/data/processed/tiles_metadata.csv'):
    !cp "/content/drive/MyDrive/diploma_data/processed_data_v2.zip" /content/
    !unzip -qo /content/processed_data_v2.zip -d /content/_data_tmp/
    !mkdir -p /content/diploma_project/data/processed
    !cp -r /content/_data_tmp/* /content/diploma_project/data/processed/
    !rm -rf /content/_data_tmp /content/processed_data_v2.zip
    print(f"Data extracted: {len(os.listdir('/content/diploma_project/data/processed'))} files")
else:
    print('Data already in place.')

!pip install -q umap-learn

---
## Обучение
Выберите **ОДНУ** из ячеек:
- **3a** — SimCLR с нуля
- **3b** — SimCLR продолжение (resume)
- **3c** — SimCLR v2 (batch=128, temp=0.2)
- **4** — BYOL

In [ ]:
# === 3a: SimCLR — первый запуск ===
!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/ \
    --epochs 50 \
    --batch_size 64 \
    --workers 4

In [ ]:
# === 3b: SimCLR — продолжение после обрыва ===
LAST_EPOCH = 29  # <-- номер последнего чекпоинта
REMAINING = 50 - LAST_EPOCH

!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/ \
    --epochs {REMAINING} \
    --batch_size 64 \
    --workers 2 \
    --resume /content/drive/MyDrive/diploma_checkpoints/simclr_resnet50_epoch_{LAST_EPOCH}.pth \
    --start_epoch {LAST_EPOCH}

In [ ]:
# === 3c: SimCLR v2 — агрессивные параметры ===
!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints_v2/ \
    --epochs 30 \
    --batch_size 80 \
    --temperature 0.2 \
    --workers 4

In [ ]:
# === 4: BYOL ===
!python /content/diploma_project/src/models/deep_clustering/train_byol.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints_byol/ \
    --epochs 30 \
    --batch_size 64 \
    --workers 4

---
## Извлечение эмбеддингов и сравнение с Baseline

После обучения нужно:
1. Извлечь эмбеддинги для SimCLR и/или BYOL
2. Запустить evaluation: cross-scale retrieval + scale invariance metrics
3. Сохранить финальные CSV на Drive

In [ ]:
# Укажи путь к лучшему чекпоинту:
CHECKPOINT = '/content/drive/MyDrive/diploma_checkpoints/simclr_resnet50_epoch_29.pth'
MODEL_TYPE = 'simclr'  # 'simclr' или 'byol'

# CHECKPOINT = '/content/drive/MyDrive/diploma_checkpoints_v2/simclr_resnet50_epoch_30.pth'
# CHECKPOINT = '/content/drive/MyDrive/diploma_checkpoints_byol/byol_resnet50_epoch_30.pth'
# MODEL_TYPE = 'byol'

!python /content/diploma_project/src/models/deep_clustering/extract_simclr_embeddings.py \
    --checkpoint {CHECKPOINT} \
    --model_type {MODEL_TYPE} \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_results/ \
    --batch_size 80 \
    --workers 4

In [ ]:
from IPython.display import Image, display
import os

umap_path = '/content/drive/MyDrive/diploma_results/baseline_vs_simclr_umap.png'
if os.path.exists(umap_path):
    display(Image(umap_path))
else:
    print(f"UMAP plot not found at {umap_path}")

---
## Evaluation: Scale Invariance + Cross-Scale Retrieval

Запуск единого evaluation pipeline после того, как эмбеддинги SimCLR/BYOL сохранены.

**Важно:** убедитесь, что файлы эмбеддингов на месте:
- `data/embeddings/resnet50_embeddings.npy` (baseline)
- `data/embeddings/simclr/finetuned_embeddings.npy`
- `data/embeddings/byol/finetuned_embeddings.npy`

In [ ]:
# === 5a: Подготовка эмбеддингов для evaluation ===
# После обучения SimCLR/BYOL и извлечения эмбеддингов,
# скопируйте файлы из Drive в правильную структуру проекта.

import os, shutil

EMB_DIR = '/content/diploma_project/data/embeddings'
RESULTS_DIR = '/content/drive/MyDrive/diploma_results'

# Baseline уже должен быть на месте
baseline = os.path.join(EMB_DIR, 'resnet50_embeddings.npy')
print(f"Baseline: {'OK' if os.path.exists(baseline) else 'MISSING'}")

# SimCLR
simclr_src = os.path.join(RESULTS_DIR, 'finetuned_embeddings.npy')
simclr_dst_dir = os.path.join(EMB_DIR, 'simclr')
os.makedirs(simclr_dst_dir, exist_ok=True)
simclr_dst = os.path.join(simclr_dst_dir, 'finetuned_embeddings.npy')
if os.path.exists(simclr_src):
    shutil.copy2(simclr_src, simclr_dst)
    print(f"SimCLR embeddings copied -> {simclr_dst}")
else:
    print(f"SimCLR embeddings not found at {simclr_src}")

# BYOL (если обучали)
byol_src = os.path.join(RESULTS_DIR, 'byol_finetuned_embeddings.npy')
byol_dst_dir = os.path.join(EMB_DIR, 'byol')
os.makedirs(byol_dst_dir, exist_ok=True)
byol_dst = os.path.join(byol_dst_dir, 'finetuned_embeddings.npy')
if os.path.exists(byol_src):
    shutil.copy2(byol_src, byol_dst)
    print(f"BYOL embeddings copied -> {byol_dst}")
else:
    print(f"BYOL embeddings not found at {byol_src} (skip if not trained)")

# Проверка
print(f"\nEmbeddings directory contents:")
for root, dirs, files in os.walk(EMB_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        print(f"  {os.path.relpath(fpath, EMB_DIR)}")

In [ ]:
# === 5b: Unified SEM Evaluation ===
# Запускает Cross-Scale Retrieval + Scale Invariance Metrics для всех моделей.
# Результаты сохраняются в /content/drive/MyDrive/diploma_results/

%cd /content/diploma_project

!python src/evaluation/run_sem_evaluation.py \
    --meta_path data/processed/tiles_metadata.csv \
    --emb_dir data/embeddings \
    --output_dir /content/drive/MyDrive/diploma_results/ \
    --K 10 \
    --n_clusters 4

In [ ]:
# === 5c: Отображение финальных результатов ===
import pandas as pd
from IPython.display import display, Image
import os

RESULTS = '/content/drive/MyDrive/diploma_results'

# Cross-Scale Retrieval
csv1 = os.path.join(RESULTS, 'cross_scale_comparison.csv')
if os.path.exists(csv1):
    print("=" * 60)
    print("CROSS-SCALE RETRIEVAL")
    print("=" * 60)
    display(pd.read_csv(csv1))
else:
    print(f"Cross-scale results not found: {csv1}")

print()

# Scale Invariance Metrics
csv2 = os.path.join(RESULTS, 'scale_invariance_metrics.csv')
if os.path.exists(csv2):
    print("=" * 60)
    print("SCALE INVARIANCE METRICS")
    print("=" * 60)
    display(pd.read_csv(csv2))
else:
    print(f"Scale invariance results not found: {csv2}")

# UMAP визуализация
umap_path = os.path.join(RESULTS, 'baseline_vs_simclr_umap.png')
if os.path.exists(umap_path):
    print("\nUMAP: Baseline vs SimCLR")
    display(Image(umap_path))

In [ ]:
# === 5d: Сохранение финальных CSV на Drive ===
# Убедитесь, что все результаты сохранены на Drive для дальнейшего анализа.

import glob

print("Files in diploma_results/:")
for f in sorted(glob.glob('/content/drive/MyDrive/diploma_results/*')):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):45s} {size_kb:8.1f} KB")

print("\nFiles in diploma_checkpoints/:")
for f in sorted(glob.glob('/content/drive/MyDrive/diploma_checkpoints/*')):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {os.path.basename(f):45s} {size_mb:8.1f} MB")

print("\nDone! All results saved to Google Drive.")